# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + methodology questions

**Source.** FlyRank, *The State of AI-Driven SEO* (March 2026), as represented in the supplied summary. The repository does not contain the paper's text, so the labels, windows and validation designs below are questions to confirm in the primary source, not verified methods. (`submission/paper_url.txt` points to my own paper, not this reference.)

**Finding 1 — growing pages reported as longer and younger** (about 3.2k vs 2.3k words; 184 vs 230 days; +37.6% words as reported). Which metric, windows and thresholds define "growing" and "declining"? How were zero or very small baselines handled, given that young pages have small baselines and can show large relative growth easily? Were these descriptive portfolio comparisons, and were results checked across brands rather than treating every page as independent?

**Finding 2 — refreshed older pages reported with higher health and impressions** (3.2x health; 57x impressions; content older than 365 days; the summary suggests a 30-day window, to be confirmed). What is the exact health measure and comparison window? Were the same pages measured before and after the refresh? If refreshed pages were compared with unrefreshed ones, how were selection into refresh, baseline differences and seasonality handled? Without a before/after or matched design, I would read this as an observed association, not an effect of refreshing.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

repo_root = next(
    (p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
     if (p / "data" / "processed" / "refresh_feature_vector.csv").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository processed feature vector")

df = pd.read_csv(repo_root / "data" / "processed" / "refresh_feature_vector.csv")
assert len(df) == 30_000, f"Unexpected row count: {len(df):,}"
assert {"client_id", "content_id", "trend_direction", "is_declining_label"}.issubset(df.columns)
assert (
    df["is_declining_label"].astype(int) == df["trend_direction"].eq("down").astype(int)
).all(), "The target does not match trend_direction == 'down'"

paper_findings = pd.DataFrame([
    {"finding": "1. Growing vs declining content",
     "reported_measure": "Mean word count; mean age",
     "reported_comparison": "~3.2k vs ~2.3k words; 184 vs 230 days",
     "reported_difference": "+37.6% words; -20% age (as reported)"},
    {"finding": "2. Refreshed older content",
     "reported_measure": "Content health; impressions",
     "reported_comparison": "Older than 365 days, after refresh (30-day window, to confirm)",
     "reported_difference": "3.2x health; 57x impressions"},
])
display(paper_findings)
print("Figures come from the supplied summary of FlyRank (March 2026); confirm definitions against the primary source.")
print(f"Data: {len(df):,} pages, {df['client_id'].nunique()} pseudonymous clients; "
      f"decline-proxy rate {df['is_declining_label'].mean():.3f}")

,finding,reported_measure,reported_comparison,reported_difference
0,1. Growing vs declining content,Mean word count; mean age,~3.2k vs ~2.3k words; 184 vs 230 days,+37.6% words; -20% age (as reported)
1,2. Refreshed older content,Content health; impressions,"Older than 365 days, after refresh (30-day win...",3.2x health; 57x impressions


Figures come from the supplied summary of FlyRank (March 2026); confirm definitions against the primary source.
Data: 30,000 pages, 32 pseudonymous clients; decline-proxy rate 0.542


## 2. My model under an honest split (before/after)

**Design.** Arm A is quoted from the W05 seed-42 client holdout (W05 allowlist and zero-fill), not re-run here. Arm B is five-fold GroupKFold by client with the W05 pipeline; arm C uses the same folds on raw values, with median imputation and missingness indicators fitted inside each training fold. B vs C changes preprocessing only, with identical test pages and no client overlap. A vs B is a contextual split comparison because A was not re-run here.

**Metrics.** CV metrics are computed per fold and averaged without weights; there is no pooled out-of-fold ranking and no row-level prediction file. The effect measure is Precision@K minus that fold's prevalence. Model-minus-rule differences are paired on the same test pages, and their SD is descriptive, not a confidence interval. Ties are checked on the primary score at each cutoff (none found). The valid-baseline subset keeps pages with `impressions_prev_30d > 0` (an evaluation filter, never a feature), re-ranks within the subset, and uses the subset prevalence as the reference. The data has no calendar dates, so nothing here tests a future outcome.

**What the outputs show.**
- The quoted W05 holdout has model Precision@50 of 0.460; arm B's five-fold mean is 0.800, a gap consistent with an evaluation-partition effect, although the W05 holdout was not re-run here. Arm C's mean is 0.792 on the same folds as B, showing little change from the preprocessing comparison.
- Model Precision@K exceeded fold prevalence in all five folds at K = 20, 50 and 100, and exceeded the rule in 4 of 5 folds at each K; fold 5 was the exception.
- Whole-fold AUC was modest (mean 0.655). Within the valid-baseline subset, AUC ranged from 0.501 to 0.627, yet Precision@50 still exceeded subset prevalence in every fold.
- Model Top-50 queues drew on 3–5 clients in folds 2–5 (largest share 0.52–0.66), while the rule's Top-50 queues were dominated by one client in every fold (share at least 0.82), so the two were compared on queues with different client mixes.

In [2]:
# ===== Q2: computation + compact outputs in one cell =====
# Requires the Q1 cell (imports, repo_root, df).
KS = (20, 50, 100)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
features = numeric_features + categorical_features

# ---- Data: raw file aligned to the processed row order ----
raw_df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
assert raw_df["content_id"].is_unique and df["content_id"].is_unique
raw_by_id = raw_df.set_index("content_id")
assert set(raw_by_id.index) == set(df["content_id"]), "Raw and processed populations differ"
raw_model_df = raw_by_id.loc[df["content_id"]].reset_index()
assert raw_model_df["trend_direction"].eq(df["trend_direction"]).all(), "Raw and processed labels do not align"
assert (raw_model_df["content_id"].to_numpy() == df["content_id"].to_numpy()).all()
position_zero_count = int(pd.to_numeric(raw_model_df["avg_position"], errors="coerce").eq(0).sum())


def clean_raw(frame):
    out = frame.copy()
    for column in numeric_features:
        if column.startswith("log_"):
            source = column.removeprefix("log_")
            out[column] = np.log1p(pd.to_numeric(out[source], errors="coerce").clip(lower=0))
        else:
            out[column] = pd.to_numeric(out[column], errors="coerce")
        out[column] = out[column].replace([np.inf, -np.inf], np.nan)
    out.loc[out["avg_position"].eq(0), "avg_position"] = np.nan  # zero means missing position
    for column in categorical_features:
        out[column] = out[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
    return out


def clean_processed(frame):
    out = frame.copy()
    for column in numeric_features:
        out[column] = pd.to_numeric(out[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
    out.loc[out["avg_position"].eq(0), "avg_position"] = np.nan  # W05 handling; other zeros stay filled
    for column in categorical_features:
        out[column] = out[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
    return out


raw_model_df = clean_raw(raw_model_df)
proc_model_df = clean_processed(df)
assert set(features).issubset(proc_model_df.columns), "Processed data lacks some allowlist features"
frames = {"B_W05_zero_fill": proc_model_df, "C_raw_missing": raw_model_df}

# ---- One shared list of client folds for both arms ----
y_all = raw_model_df["trend_direction"].eq("down").astype(int)
groups = raw_model_df["client_id"].astype(str)
assert (df["client_id"].astype(str).to_numpy() == groups.to_numpy()).all()
splits = list(GroupKFold(n_splits=5).split(raw_model_df, y_all, groups=groups))
for train_idx, test_idx in splits:
    assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])), "A client crossed train/test"
assert np.array_equal(np.sort(np.concatenate([t for _, t in splits])), np.arange(len(df)))


# ---- Model, rule, queues ----
def make_logistic_model():
    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scaler", StandardScaler())]), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([("preprocessor", preprocessor),
                     ("model", LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42))])


def week4_scores(frame):
    imp = pd.to_numeric(frame["impressions_90d"], errors="coerce")
    return imp.where(frame["content_age_days"].ge(180) & imp.ge(500), 0.0).to_numpy()


def build_queue(frame, scores, method):
    q = pd.DataFrame({
        "content_id": frame["content_id"].to_numpy(),
        "client_id": frame["client_id"].astype(str).to_numpy(),
        "y": frame["y"].to_numpy(),
        "score": np.asarray(scores, dtype=float),
        "impressions": pd.to_numeric(frame["impressions_90d"], errors="coerce").to_numpy(),
        "age": frame["content_age_days"].to_numpy(),
    })
    if method == "rule":   # Week-4 order: score, impressions, age (desc), then content_id (asc)
        keys, ascending = ["score", "impressions", "age", "content_id"], [False, False, False, True]
    else:                  # model: score, then content_id
        keys, ascending = ["score", "content_id"], [False, True]
    return q.sort_values(keys, ascending=ascending, kind="mergesort").reset_index(drop=True)


def topk_precision(queue, k):
    return float(queue["y"].head(k).mean())


def tie_row(queue, k):  # ties on the primary score at the cutoff, before tie-breakers
    scores = queue["score"].to_numpy()
    cutoff = scores[min(k, len(scores)) - 1]
    in_queue = np.arange(len(scores)) < k
    exact = scores == cutoff
    near = np.isclose(scores, cutoff, rtol=1e-9, atol=1e-12)
    return {"cutoff_score": float(cutoff),
            "exact_ties": int(exact.sum() - 1), "exact_excluded": int((exact & ~in_queue).sum()),
            "isclose_ties": int(near.sum() - 1), "isclose_excluded": int((near & ~in_queue).sum())}


def composition_row(queue, k):
    share = queue["client_id"].head(k).value_counts(normalize=True)
    return {"clients_represented": int(share.size), "largest_client_share": float(share.iloc[0])}


# ---- Fold loop (both arms, identical folds; OOF predictions are not stored) ----
fold_rows, tie_rows, comp_rows, valid_rows = [], [], [], []
for arm, frame in frames.items():
    X_all = frame[features]
    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        model = make_logistic_model().fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
        test = raw_model_df.iloc[test_idx].assign(y=y_all.iloc[test_idx].to_numpy())
        y_test = test["y"].to_numpy()
        model_scores = model.predict_proba(X_all.iloc[test_idx])[:, 1]
        rule_scores = week4_scores(test)
        prevalence = float(y_test.mean())
        queues = {"model": build_queue(test, model_scores, "model"),
                  "rule": build_queue(test, rule_scores, "rule")}

        row = {"pipeline": arm, "fold": fold_number,
               "test_clients": int(groups.iloc[test_idx].nunique()), "test_pages": len(test_idx),
               "prevalence": prevalence,
               "model_ROC_AUC": float(roc_auc_score(y_test, model_scores)),
               "model_AP": float(average_precision_score(y_test, model_scores))}
        for method, queue in queues.items():
            for k in KS:
                row[f"{method}_P@{k}"] = topk_precision(queue, k)
                key = {"pipeline": arm, "fold": fold_number, "method": method, "K": k}
                tie_rows.append({**key, **tie_row(queue, k)})
                comp_rows.append({**key, "queue_size": k, **composition_row(queue, k)})
        for k in KS:
            row[f"model_abs_lift@{k}"] = row[f"model_P@{k}"] - prevalence
            row[f"rule_abs_lift@{k}"] = row[f"rule_P@{k}"] - prevalence
            row[f"paired_model_minus_rule@{k}"] = row[f"model_P@{k}"] - row[f"rule_P@{k}"]
        fold_rows.append(row)

        # Valid-baseline subset: previous-30-day impressions > 0 (evaluation filter only)
        valid = pd.to_numeric(test["impressions_prev_30d"], errors="coerce").gt(0).to_numpy()
        sub, sub_y = test.loc[valid], y_test[valid]
        sub_prev = float(sub_y.mean())
        vrow = {"pipeline": arm, "fold": fold_number, "test_pages": len(test),
                "excluded_zero_or_missing_baseline": int((~valid).sum()),
                "valid_subset_pages": int(valid.sum()), "subset_prevalence": sub_prev}
        for method, scores in (("model", model_scores[valid]), ("rule", rule_scores[valid])):
            vrow[f"{method}_ROC_AUC"] = float(roc_auc_score(sub_y, scores))
            vrow[f"{method}_AP"] = float(average_precision_score(sub_y, scores))
            vrow[f"{method}_AP_minus_prevalence"] = vrow[f"{method}_AP"] - sub_prev
            sub_queue = build_queue(sub, scores, method)
            for k in KS:
                vrow[f"{method}_P@{k}"] = topk_precision(sub_queue, k)
                vrow[f"{method}_abs_lift@{k}"] = vrow[f"{method}_P@{k}"] - sub_prev
        valid_rows.append(vrow)

fold_metrics = pd.DataFrame(fold_rows)
tie_diagnostics = pd.DataFrame(tie_rows)
queue_composition = pd.DataFrame(comp_rows)
valid_subset_metrics = pd.DataFrame(valid_rows)

# W05 figures are references quoted from the separate W05 notebook, not re-run here.
W05_QUOTED = {"prevalence": 0.390968, "model_P@20": 0.40, "model_P@50": 0.46, "model_P@100": 0.49,
              "rule_P@20": 0.35, "rule_P@50": 0.30, "rule_P@100": 0.33,
              "model_ROC_AUC": 0.699716, "model_AP": 0.52946}
W05_VALID_QUOTED = {"pages": 1499, "prevalence": 0.606404, "model_ROC_AUC": 0.610546, "model_AP": 0.702}
w05_source = "quoted from the W05 notebook; not re-run in this cell"
w05_metrics = W05_QUOTED
w05_valid_reference = W05_VALID_QUOTED
for key, value in {"model_P@20": 0.400, "model_P@50": 0.460, "model_P@100": 0.490,
                   "model_ROC_AUC": 0.700, "model_AP": 0.529}.items():
    assert abs(w05_metrics[key] - value) < 1e-3, f"W05 quoted reference mismatch for {key}"

# ---- Checks ----
B = fold_metrics[fold_metrics["pipeline"].eq("B_W05_zero_fill")].sort_values("fold").reset_index(drop=True)
C = fold_metrics[fold_metrics["pipeline"].eq("C_raw_missing")].sort_values("fold").reset_index(drop=True)
assert np.allclose(B[["test_clients", "test_pages", "prevalence"]], C[["test_clients", "test_pages", "prevalence"]])
tie_cols = ["exact_ties", "exact_excluded", "isclose_ties", "isclose_excluded"]
assert tie_diagnostics[tie_cols].to_numpy().max() == 0, "Ties found at a cutoff; inspect tie_diagnostics"
assert tie_diagnostics.loc[tie_diagnostics["method"].eq("rule"), "cutoff_score"].gt(0).all()

# ---- Output 1: three-arm summary ----
metric_cols = ["prevalence", "model_P@20", "model_P@50", "model_P@100",
               "rule_P@20", "rule_P@50", "rule_P@100", "model_ROC_AUC", "model_AP"]
three_arm = pd.DataFrame({
    "A: W05 holdout (quoted)": pd.Series({c: w05_metrics[c] for c in metric_cols}),
    "B: 5-fold, W05 zero-fill": B[metric_cols].mean(),
    "C: 5-fold, raw missing": C[metric_cols].mean(),
}).T
print(f"Arm A is {w05_source}; B and C are unweighted means of per-fold metrics (no pooled OOF ranking).")
display(three_arm.round(3))

# ---- Output 2: arm C by fold (precision, queue composition, valid-baseline subset) ----
top50 = queue_composition[queue_composition["pipeline"].eq("C_raw_missing") & queue_composition["K"].eq(50)]
comp = top50.pivot(index="fold", columns="method", values=["clients_represented", "largest_client_share"])
comp.columns = [f"{method}_{value}" for value, method in comp.columns]
comp = comp.reset_index()
Vc = valid_subset_metrics[valid_subset_metrics["pipeline"].eq("C_raw_missing")].sort_values("fold")
by_fold = (
    C[["fold", "test_clients", "prevalence", "model_P@50", "rule_P@50", "model_abs_lift@50",
       "paired_model_minus_rule@50"]]
    .merge(comp[["fold", "model_clients_represented", "model_largest_client_share",
                 "rule_largest_client_share"]], on="fold")
    .merge(Vc[["fold", "model_ROC_AUC", "model_abs_lift@50"]]
           .rename(columns={"model_ROC_AUC": "valid_AUC", "model_abs_lift@50": "valid_lift@50"}), on="fold")
    .rename(columns={"model_abs_lift@50": "model_lift@50",
                     "paired_model_minus_rule@50": "model_minus_rule@50",
                     "model_clients_represented": "model_top50_clients",
                     "model_largest_client_share": "model_top50_share",
                     "rule_largest_client_share": "rule_top50_share"})
)
print("Arm C by fold at K = 50 (lift = Precision@50 minus prevalence; top50_* = queue composition; "
      "valid_* = valid-baseline subset with its own prevalence):")
display(by_fold.round(3))

# ---- Output 3: short summary lines ----
for k in KS:
    paired = C[f"paired_model_minus_rule@{k}"]
    model_mean, rule_mean = C[f"model_P@{k}"].mean(), C[f"rule_P@{k}"].mean()
    above_prevalence = int((C[f"model_abs_lift@{k}"] > 0).sum())
    above_rule = int((paired > 0).sum())
    assert above_prevalence == 5 and above_rule == 4
    assert (paired.le(0).to_numpy() == C["fold"].eq(5).to_numpy()).all()
    print(f"K={k}: mean model {model_mean:.3f} vs rule {rule_mean:.3f}; model above prevalence in "
          f"{above_prevalence}/5 folds and above the rule in {above_rule}/5; mean paired difference "
          f"{paired.mean():.3f} (fold SD {paired.std(ddof=1):.3f}, descriptive, not a CI).")
auc, vauc = C["model_ROC_AUC"], Vc["model_ROC_AUC"]
print(f"Model ROC AUC by fold: mean {auc.mean():.3f}, range {auc.min():.3f}-{auc.max():.3f}. "
      f"Valid-baseline subset: mean {vauc.mean():.3f}, range {vauc.min():.3f}-{vauc.max():.3f}.")
print(f"W05 one-holdout valid-baseline reference ({w05_source}): {int(w05_valid_reference['pages'])} pages, "
      f"prevalence {w05_valid_reference['prevalence']:.3f}, AUC {w05_valid_reference['model_ROC_AUC']:.3f}, "
      f"AP {w05_valid_reference['model_AP']:.3f}.")
print(f"Cutoff ties on the primary score ({len(tie_diagnostics)} checks): none found; all rule cutoff scores "
      f"are positive, so the impressions tie-breaker repeats the primary score.")
print(f"avg_position zero codes treated as missing: {position_zero_count:,}. "
      "No row-level prediction file was written.")

# ---- Tie the Markdown numbers to this cell ----
assert round(B["model_P@50"].mean(), 3) == 0.800 and round(C["model_P@50"].mean(), 3) == 0.792
assert round(w05_metrics["model_P@50"], 3) == 0.460
assert round(auc.mean(), 3) == 0.655
assert (round(vauc.min(), 3), round(vauc.max(), 3)) == (0.501, 0.627)
assert (Vc["model_abs_lift@50"] > 0).all()
folds_2_5 = comp[comp["fold"].ge(2)]
assert (folds_2_5["model_clients_represented"].min(), folds_2_5["model_clients_represented"].max()) == (3, 5)
assert (round(folds_2_5["model_largest_client_share"].min(), 2),
        round(folds_2_5["model_largest_client_share"].max(), 2)) == (0.52, 0.66)
assert round(comp["rule_largest_client_share"].min(), 2) == 0.82

Arm A is quoted from the W05 notebook; not re-run in this cell; B and C are unweighted means of per-fold metrics (no pooled OOF ranking).


,prevalence,model_P@20,model_P@50,model_P@100,rule_P@20,rule_P@50,rule_P@100,model_ROC_AUC,model_AP
A: W05 holdout (quoted),0.391,0.40,0.460,0.490,0.35,0.30,0.330,0.700,0.529
"B: 5-fold, W05 zero-fill",0.544,0.76,0.800,0.768,0.53,0.54,0.542,0.656,0.665
"C: 5-fold, raw missing",0.544,0.76,0.792,0.774,0.53,0.54,0.542,0.655,0.663


Arm C by fold at K = 50 (lift = Precision@50 minus prevalence; top50_* = queue composition; valid_* = valid-baseline subset with its own prevalence):


,fold,test_clients,prevalence,model_P@50,rule_P@50,model_lift@50,model_minus_rule@50,model_top50_clients,model_top50_share,rule_top50_share,valid_AUC,valid_lift@50
0,1,1,0.490,0.74,0.40,0.250,0.34,1.0,1.00,1.00,0.611,0.248
1,2,7,0.645,0.90,0.48,0.255,0.42,5.0,0.66,1.00,0.607,0.234
2,3,8,0.379,0.66,0.38,0.281,0.28,5.0,0.52,1.00,0.627,0.267
3,4,8,0.622,0.84,0.60,0.218,0.24,4.0,0.54,0.82,0.501,0.143
4,5,8,0.585,0.82,0.84,0.235,-0.02,3.0,0.52,0.94,0.576,0.137


K=20: mean model 0.760 vs rule 0.530; model above prevalence in 5/5 folds and above the rule in 4/5; mean paired difference 0.230 (fold SD 0.199, descriptive, not a CI).
K=50: mean model 0.792 vs rule 0.540; model above prevalence in 5/5 folds and above the rule in 4/5; mean paired difference 0.252 (fold SD 0.166, descriptive, not a CI).
K=100: mean model 0.774 vs rule 0.542; model above prevalence in 5/5 folds and above the rule in 4/5; mean paired difference 0.232 (fold SD 0.192, descriptive, not a CI).
Model ROC AUC by fold: mean 0.655, range 0.612-0.718. Valid-baseline subset: mean 0.584, range 0.501-0.627.
W05 one-holdout valid-baseline reference (quoted from the W05 notebook; not re-run in this cell): 1499 pages, prevalence 0.606, AUC 0.611, AP 0.702.
Cutoff ties on the primary score (60 checks): none found; all rule cutoff scores are positive, so the impressions tie-breaker repeats the primary score.
avg_position zero codes treated as missing: 1,205. No row-level prediction file

## 3. Leakage audit

The 26-feature allowlist excludes the target and its direct sources (`trend_direction`, `trend_pct`), IDs, `measurable_opportunity`, and all last/previous-30-day columns, including the `impressions_prev_30d` baseline that is merged only for evaluation. The exclusion assertions are re-run after that merge. IDs are used only for grouping, joining and deterministic tie-breaks. In W03 (quoted, not recomputed here), deliberately leaking `trend_pct` raised AUC to 0.9996 versus 0.709 for the clean random-row feature set, which is why the direct source stays out.

**Temporal overlap remains.** Trailing-90-day activity, rate and engagement features include the latest-30-day label window, so the results are within-window rankings of a current proxy, not predictions of future decline, and no calendar dates exist for a chronological split. The shared prep script zero-filled many numeric missing values; the W06 evaluation instead reads raw values and imputes inside each training fold. That is a preprocessing correction, not evidence of direct target leakage.

**Not tested here (limits, not proven defects).** The provenance and cut points of the tier columns; whether `days_since_last_update` reflects update timing, data-batch practice or client differences; whether missingness indicators partly encode client identity (the zero-fill versus raw-missing comparison shows that handling changed results very little, but it does not test this, because zero-filled values still mark missing entries); and the contribution of each feature group, since no ablation was run.

In [3]:
forbidden_features = {
    "content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label",
    "measurable_opportunity",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
assert "impressions_prev_30d" in raw_model_df.columns, "Raw evaluation baseline was not merged"
assert "impressions_prev_30d" not in features, "Evaluation-only baseline entered the feature allowlist"
assert set(features).isdisjoint(forbidden_features), (
    f"Forbidden columns are model inputs: {set(features) & forbidden_features}"
)
assert set(features).issubset(raw_model_df.columns)
assert raw_model_df["content_id"].is_unique
assert (
    raw_model_df["trend_direction"].eq("down").astype(int).to_numpy()
    == raw_model_df["content_id"].map(df.set_index("content_id")["is_declining_label"]).to_numpy()
).all(), "The audited target no longer matches its documented source"

missing = raw_model_df[features].isna().sum()
missing = missing[missing.gt(0)].sort_values(ascending=False)
print(f"Allowlist: {len(features)} features; forbidden-feature assertions passed after the raw-baseline merge.")
print(f"Features with source missing values ({len(missing)} of {len(features)}): "
      + ", ".join(f"{name} {count:,}" for name, count in missing.items()))
print("Window audit: trailing-90-day activity/rate features overlap the label's latest-30-day window; "
      "strictly pre-label timing is not established.")
print("ID audit: content_id and client_id are absent from the allowlist; client_id is used only for grouped splitting.")

Allowlist: 26 features; forbidden-feature assertions passed after the raw-baseline merge.
Features with source missing values (7 of 26): word_count 7,699, char_count 7,699, search_volume 2,468, cpc 2,468, competition 2,468, avg_position 1,205, scroll_rate 125
Window audit: trailing-90-day activity/rate features overlap the label's latest-30-day window; strictly pre-label timing is not established.
ID audit: content_id and client_id are absent from the allowlist; client_id is used only for grouped splitting.


## 4. Claim rewrite

**Careful claim (observed, measured, directional, decision-support).** In five client-grouped folds, the Logistic Regression score had higher Precision@50 than the Week-4 rule in 4 of 5 held-out folds (mean 0.792 vs 0.540; mean fold prevalence 0.544; fold 5 was the exception) and exceeded each fold's prevalence by 0.22–0.28. Whole-fold discrimination was modest (mean ROC AUC 0.655; 0.50–0.63 among pages with a positive previous-30-day baseline), results depend on which clients are held out (quoted W05 holdout: 0.460), and the rule's Top-50 queues were dominated by one client (largest share at least 0.82). These are within-window rankings of a current decline proxy that may help prioritize pages for human review; they do not show future decline, causation, or calibrated probabilities, and precision beyond K = 100 and feature-group contributions were not tested.

In [4]:
C = fold_metrics[fold_metrics["pipeline"].eq("C_raw_missing")].sort_values("fold").reset_index(drop=True)
V = valid_subset_metrics[valid_subset_metrics["pipeline"].eq("C_raw_missing")]
top50 = queue_composition[queue_composition["pipeline"].eq("C_raw_missing") & queue_composition["K"].eq(50)]
rule_share = top50.loc[top50["method"].eq("rule"), "largest_client_share"]
paired50 = C["paired_model_minus_rule@50"]

evidence = {
    "mean_model_P@50": C["model_P@50"].mean(),
    "mean_rule_P@50": C["rule_P@50"].mean(),
    "mean_fold_prevalence": C["prevalence"].mean(),
    "folds_model_above_rule_P@50": int((paired50 > 0).sum()),
    "fold5_model_minus_rule_P@50": float(paired50[C["fold"].eq(5)].iloc[0]),
    "model_lift_P@50_min": C["model_abs_lift@50"].min(),
    "model_lift_P@50_max": C["model_abs_lift@50"].max(),
    "mean_AUC_full_folds": C["model_ROC_AUC"].mean(),
    "AUC_valid_baseline_min": V["model_ROC_AUC"].min(),
    "AUC_valid_baseline_max": V["model_ROC_AUC"].max(),
    "rule_top50_min_largest_client_share": rule_share.min(),
    "W05_holdout_model_P@50": w05_metrics["model_P@50"],
}

# Every number in the claim text must match a cell value
assert (C["model_abs_lift@50"] > 0).all()
assert round(evidence["mean_model_P@50"], 3) == 0.792
assert round(evidence["mean_rule_P@50"], 3) == 0.540
assert round(evidence["mean_fold_prevalence"], 3) == 0.544
assert evidence["folds_model_above_rule_P@50"] == 4 and evidence["fold5_model_minus_rule_P@50"] < 0
assert (round(evidence["model_lift_P@50_min"], 2), round(evidence["model_lift_P@50_max"], 2)) == (0.22, 0.28)
assert round(evidence["mean_AUC_full_folds"], 3) == 0.655
assert (round(evidence["AUC_valid_baseline_min"], 2), round(evidence["AUC_valid_baseline_max"], 2)) == (0.50, 0.63)
assert round(evidence["rule_top50_min_largest_client_share"], 2) == 0.82
assert round(evidence["W05_holdout_model_P@50"], 3) == 0.460

print("Numbers used in the claim (all asserted against the text above):")
display(pd.Series(evidence).round(3).to_frame("value"))
print("These are within-window rankings of a current proxy; no pooled OOF ranking is used.")

Numbers used in the claim (all asserted against the text above):


,value
mean_model_P@50,0.792
mean_rule_P@50,0.540
mean_fold_prevalence,0.544
folds_model_above_rule_P@50,4.000
fold5_model_minus_rule_P@50,-0.020
model_lift_P@50_min,0.218
model_lift_P@50_max,0.281
mean_AUC_full_folds,0.655
AUC_valid_baseline_min,0.501
AUC_valid_baseline_max,0.627


These are within-window rankings of a current proxy; no pooled OOF ranking is used.
